# z027 - Reprueba de B y C de z026, versión "limpia", con un tercer corte

z026: B y C ganaron claro en junio (+10,7 / +12,3 M en N fijo) pero empataron en mayo, donde el train (202103) no tiene historia.
Acá se prueban sin lo que tuvo gain 0 y con un **tercer corte que sí tiene historia en el train**.

| Versión | Variables |
|---|---|
| ref | receta actual (761) |
| C_limpio | + grupo C (17) + delta 1 de c_trx_total, c_servicios_usados, c_trx_digitales (sin delta 2) = 20 |
| BC_limpio | + C limpio + grupo B sin c_cheques_rechazados (12) + lag 1 y delta 1 de sus 11 montos (sin lag2/delta2) = 54 |

Cortes (10 semillas, trial 34, target juntar, ganancia con BAJA+2):
- **junio**: train 202104 → valid 202106 · **mayo**: train 202103 → valid 202105 · **jun2m**: train 202103 + 202104 → valid 202106 (como z010).

**Regla (definida antes de correr, 11/10):** se adopta si, contra la referencia,
en **junio** y en **jun2m** gana en ≥ 8/10 semillas (ganancia máx) **y** mejora la ganancia media en N fijo; y en **mayo** no pierde
más de 2 M en N fijo. Ojo: el corte de selección es junio, ya mirado en z026 (las variables "limpias" se eligieron con ese resultado),
y jun2m también valida en junio → no es un mes nuevo.

**Versión para el submit 33 (definido antes):** la que pase; si pasan las dos o ninguna, la de mayor suma de diferencias en N fijo
en los tres cortes (Keila quiere verla en el bot igual).

In [1]:
# --- 1. Librerías y parámetros ---
import os, time, shutil
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import wilcoxon

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
SEMILLAS = [270131, 417293, 583117, 692347, 851071,
            135791, 246809, 314159, 527183, 964201]
CORTES = {"junio": ([202104], 202106), "mayo": ([202103], 202105), "jun2m": ([202103, 202104], 202106)}
PROP_N = 12_000 / 164_647
CARPETA, CARPETA_Z026 = "probs_z027", "probs_z026"
os.makedirs(CARPETA, exist_ok=True)

params_fijos = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
}
elegidos = pd.read_csv("z005_top10_final.csv", sep=";", decimal=",", encoding="utf-8-sig", index_col="trial")
trial = elegidos.loc[34]
params = {**params_fijos, "num_leaves": int(trial["params_num_leaves"]),
          "min_sum_hessian_in_leaf": float(trial["params_min_sum_hessian_in_leaf"])}
N_ARBOLES = int(trial["user_attrs_best_iter"])

def log(texto):
    with open("z027_progreso.txt", "a", encoding="utf-8") as f:
        f.write(time.strftime("%H:%M:%S ") + texto + "\n")
log(f"inicio | árboles {N_ARBOLES}")

In [2]:
# --- 2. Datos (202103-202106) ---
df = pd.read_parquet("../competencia_01.parquet")
df = df[df["foto_mes"] <= 202106]
df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
df["y_juntar"] = df["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(int)
no_usar = ["foto_mes", "clase_ternaria", "y_juntar"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]   # las originales

In [3]:
# --- Variables nuevas de z026 (versión "limpia"), ANTES de los lags. df tiene que estar ordenado por cliente y mes ---
def s(*columnas):
    # Suma con fillna(0) en cada término: si no tiene Master, sus columnas son NaN y la suma daría NaN
    total = 0
    for c in columnas:
        total = total + (df[c].fillna(0) if isinstance(c, str) else c.fillna(0))
    return total

def div(a, b):
    # Cociente: si el denominador es 0 o negativo, NaN. Ej.: 1.000 dólares / saldo −5.000 → NaN (no tiene sentido)
    return (a / b.where(b > 0)).astype("float32")

n = {}
# Auxiliares de tarjetas (las usa el grupo C; el grupo A no entra)
n["c_tarjetas_trx"]      = s("ctarjeta_visa_transacciones", "ctarjeta_master_transacciones")

# ---------- Grupo B: plata del cliente (sin c_cheques_rechazados, gain 0 en z026) ----------
n["m_saldo_paquete"]     = s("mcuenta_corriente", "mcaja_ahorro")
n["m_saldo_adicional"]   = s("mcuenta_corriente_adicional", "mcaja_ahorro_adicional")
n["r_saldo_dolares"]     = div(df["mcaja_ahorro_dolares"], df["mcuentas_saldo"])
n["m_inversiones_total"] = s("mplazo_fijo_pesos", "mplazo_fijo_dolares", "minversion1_pesos", "minversion1_dolares", "minversion2")
n["m_patrimonio_banco"]  = s("mcuentas_saldo", n["m_inversiones_total"])
n["m_forex_neto"]        = df["mforex_buy"].fillna(0) - df["mforex_sell"].fillna(0)
n["m_payroll_total"]     = s("mpayroll", "mpayroll2")
n["m_ingresos_total"]    = s(n["m_payroll_total"], "mtransferencias_recibidas", "mcheques_depositados")
n["m_egresos_total"]     = s("mtransferencias_emitidas", "mcheques_emitidos", "mextraccion_autoservicio",
                             "mpagodeservicios", "mpagomiscuentas", "mcuenta_debitos_automaticos")
n["m_flujo_neto"]        = n["m_ingresos_total"] - n["m_egresos_total"]
n["m_debitos_aut_total"] = s("mcuenta_debitos_automaticos", "mttarjeta_visa_debitos_automaticos", "mttarjeta_master_debitos_automaticos")
n["m_descuentos_total"]  = s("mcajeros_propios_descuentos", "mtarjeta_visa_descuentos", "mtarjeta_master_descuentos")

# ---------- Grupo C: uso y fidelidad ----------
n["c_inversiones_total"]  = s("cplazo_fijo", "cinversion1", "cinversion2")
n["c_seguros_total"]      = s("cseguro_vida", "cseguro_auto", "cseguro_vivienda", "cseguro_accidentes_personales")
n["c_payroll_total"]      = s("cpayroll_trx", "cpayroll2_trx")
n["tiene_payroll"]        = (n["c_payroll_total"] > 0).astype("int8")
n["c_debitos_aut_total"]  = s("ccuenta_debitos_automaticos", "ctarjeta_visa_debitos_automaticos", "ctarjeta_master_debitos_automaticos")
n["c_pagos_servicios_total"] = s("cpagodeservicios", "cpagomiscuentas", n["c_debitos_aut_total"])
n["c_descuentos_total"]   = s("ccajeros_propios_descuentos", "ctarjeta_visa_descuentos", "ctarjeta_master_descuentos")
n["c_comisiones_total"]   = s("ccomisiones_mantenimiento", "ccomisiones_otras")
n["c_trx_digitales"]      = s("chomebanking_transacciones", "cmobile_app_trx")     # la app vale 0/1: suma 1 si la usó
n["c_trx_presenciales"]   = s("ccajas_transacciones", "catm_trx", "catm_trx_other")
n["r_atm_otros"]          = div(df["catm_trx_other"].fillna(0), s("catm_trx", "catm_trx_other"))
n["c_trx_total"]          = s(n["c_trx_digitales"], n["c_trx_presenciales"], "ctarjeta_debito_transacciones",
                              n["c_tarjetas_trx"], "ccallcenter_transacciones")   # el call center cuenta como actividad
usados = [df[c] for c in ["ctarjeta_debito_transacciones", "cpagodeservicios", "cpagomiscuentas", "chomebanking_transacciones",
                          "cmobile_app_trx", "catm_trx", "ccajas_transacciones", "ctransferencias_emitidas",
                          "ctransferencias_recibidas", "cforex"]] + \
         [n[c] for c in ["c_tarjetas_trx", "c_payroll_total", "c_debitos_aut_total", "m_inversiones_total", "c_seguros_total"]]
n["c_servicios_usados"]   = sum((x > 0).astype("int8") for x in usados)            # 0 a 15 (NaN cuenta como no usado)
n["r_uso_vs_tenencia"]    = div(n["c_servicios_usados"].astype("float32"), df["cproductos"])
n["banco_principal"]      = ((n["tiene_payroll"] == 1) & (n["c_debitos_aut_total"] > 0) & (n["c_tarjetas_trx"] > 0)).astype("int8")

cliente = df["numero_de_cliente"]
tp = n["tiene_payroll"]
n["meses_con_payroll_3"] = (tp + tp.groupby(cliente).shift(1).fillna(0) + tp.groupby(cliente).shift(2).fillna(0)).astype("int8")
# Racha de meses seguidos activo: cada mes inactivo abre un "bloque" nuevo; dentro del bloque se suman los meses activos
activo = (n["c_servicios_usados"] > 0).astype("int16")
bloque = (activo == 0).groupby(cliente).cumsum()
n["meses_activo_seguidos"] = activo.groupby([cliente, bloque]).cumsum().astype("int8")

nuevas_df = pd.DataFrame(n).astype("float32")
del n, usados

GRUPO_B = ["m_saldo_paquete", "m_saldo_adicional", "r_saldo_dolares", "m_patrimonio_banco", "m_inversiones_total", "m_forex_neto",
           "m_payroll_total", "m_ingresos_total", "m_egresos_total", "m_flujo_neto", "m_debitos_aut_total", "m_descuentos_total"]
GRUPO_C = ["c_inversiones_total", "c_seguros_total", "c_payroll_total", "tiene_payroll", "c_debitos_aut_total",
           "c_pagos_servicios_total", "c_descuentos_total", "c_comisiones_total", "c_trx_digitales", "c_trx_presenciales",
           "r_atm_otros", "c_trx_total", "c_servicios_usados", "r_uso_vs_tenencia", "banco_principal",
           "meses_con_payroll_3", "meses_activo_seguidos"]
M_CON_LAG1   = [c for c in GRUPO_B if c.startswith("m_")]                  # 11 montos: lag 1 y delta 1 (sin lag2/delta2: gain 0)
C_CON_DELTA1 = ["c_trx_total", "c_servicios_usados", "c_trx_digitales"]    # solo delta 1
g_nuevas = nuevas_df.groupby(df["numero_de_cliente"])
historia = {}
lag_m = g_nuevas[M_CON_LAG1].shift(1)
for c in M_CON_LAG1:
    historia[f"{c}_lag1"] = lag_m[c]
    historia[f"{c}_delta1"] = nuevas_df[c] - lag_m[c]
lag_c = g_nuevas[C_CON_DELTA1].shift(1)
for c in C_CON_DELTA1:
    historia[f"{c}_delta1"] = nuevas_df[c] - lag_c[c]
historia = pd.DataFrame(historia).astype("float32")

C_LIMPIO  = GRUPO_C + [f"{c}_delta1" for c in C_CON_DELTA1]                                      # 20 variables
B_LIMPIO  = GRUPO_B + [f"{c}_{t}1" for c in M_CON_LAG1 for t in ["lag", "delta"]]                 # 34 variables
BC_LIMPIO = B_LIMPIO + C_LIMPIO                                                                    # 54 variables
nuevas_df = pd.concat([nuevas_df, historia], axis=1)[BC_LIMPIO]       # me quedo solo con las que se usan
del historia, g_nuevas, lag_m, lag_c
print(f"C limpio: {len(C_LIMPIO)}  B+C limpio: {len(BC_LIMPIO)}")

C limpio: 20  B+C limpio: 54


In [4]:
# --- 4. Lags y deltas de las originales (como siempre) y unir las nuevas ---
grupo = df.groupby("numero_de_cliente")[cols]
partes = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    partes += [lag, delta]
df = pd.concat([df] + partes, axis=1)
del partes, lag, delta, grupo
base = [c for c in df.columns if c not in no_usar]    # las 761 de siempre
df = pd.concat([df, nuevas_df], axis=1)
del nuevas_df
VERSIONES = {"ref": base, "C_limpio": base + C_LIMPIO, "BC_limpio": base + BC_LIMPIO}
for v, vs in VERSIONES.items():
    print(v, len(vs))
log("variables listas")

ref 761
C_limpio 781
BC_limpio 815


In [5]:
# --- 5. Entrenar. ref de junio y mayo: se reusa z026 si la 1ª semilla da idéntico ---
foto = df["foto_mes"].values

def entrenar(version, cortes, semillas=SEMILLAS):
    vs = VERSIONES[version]
    for corte in cortes:
        meses_train, mes_valid = CORTES[corte]
        faltan = [s for s in semillas if not os.path.exists(f"{CARPETA}/{version}_{corte}_{s}.npy")]
        if not faltan:
            continue
        tr, va = np.isin(foto, meses_train), foto == mes_valid
        datos = lgb.Dataset(df.loc[tr, vs], label=df.loc[tr, "y_juntar"])
        X_valid = df.loc[va, vs]
        for s in faltan:
            inicio = time.time()
            modelo = lgb.train({**params, "seed": s}, datos, num_boost_round=N_ARBOLES)
            np.save(f"{CARPETA}/{version}_{corte}_{s}.npy", modelo.predict(X_valid).astype("float32"))
            log(f"{corte} {version} semilla {s}: {time.time() - inicio:.0f} s")
        del datos, X_valid

s0 = SEMILLAS[0]
for corte in ["junio", "mayo"]:
    if all(os.path.exists(f"{CARPETA}/ref_{corte}_{s}.npy") for s in SEMILLAS):
        continue
    entrenar("ref", [corte], semillas=[s0])
    iguales = np.array_equal(np.load(f"{CARPETA}/ref_{corte}_{s0}.npy"), np.load(f"{CARPETA_Z026}/ref_{corte}_{s0}.npy"))
    log(f"verificación ref {corte}: {'idéntico a z026' if iguales else 'DISTINTO → se reentrena'}")
    if iguales:
        for s in SEMILLAS[1:]:
            shutil.copy(f"{CARPETA_Z026}/ref_{corte}_{s}.npy", f"{CARPETA}/ref_{corte}_{s}.npy")
for version in VERSIONES:
    entrenar(version, list(CORTES))
log("entrenamiento terminado")

In [6]:
# --- 6. Ganancias por semilla, comparación pareada y regla ---
filas, curvas = [], []
for corte, (meses_train, mes_valid) in CORTES.items():
    va = foto == mes_valid
    gan = np.where(df.loc[va, "clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
    n_fijo = round(PROP_N * va.sum())
    for version in VERSIONES:
        acums = []
        for s in SEMILLAS:
            acum = np.cumsum(gan[np.argsort(-np.load(f"{CARPETA}/{version}_{corte}_{s}.npy"))])
            acums.append(acum)
            filas.append({"corte": corte, "version": version, "semilla": s, "gan_max": acum.max() / 1e6,
                          "mejor_n": acum.argmax() + 1, "gan_nfijo": acum[n_fijo - 1] / 1e6, "gan_12000": acum[11_999] / 1e6})
        prom = np.mean(acums, axis=0)          # método del tutor: curva promedio de las 10 semillas
        curvas.append({"version": version, "corte": corte, "max_curva_prom": prom.max() / 1e6, "n_max": prom.argmax() + 1})
por_semilla = pd.DataFrame(filas)
por_semilla.to_csv("z027_por_semilla.csv", sep=";", decimal=",", index=False, encoding="utf-8-sig")

idx = por_semilla.set_index(["version", "corte", "semilla"]).sort_index()
res = []
for version in VERSIONES:
    for corte in CORTES:
        g, r = idx.loc[(version, corte)], idx.loc[("ref", corte)]
        d_max, d_fijo = g["gan_max"] - r["gan_max"], g["gan_nfijo"] - r["gan_nfijo"]
        es_ref = version == "ref"
        res.append({"version": version, "corte": corte, "gan_max": g["gan_max"].mean(), "gan_max_desv": g["gan_max"].std(),
                    "gan_nfijo": g["gan_nfijo"].mean(), "gan_12000": g["gan_12000"].mean(), "mejor_n": g["mejor_n"].mean(),
                    "dif_max": d_max.mean(), "gana_max": np.nan if es_ref else (d_max > 0).sum(),
                    "p_max": np.nan if es_ref else wilcoxon(d_max).pvalue,
                    "dif_nfijo": d_fijo.mean(), "gana_nfijo": np.nan if es_ref else (d_fijo > 0).sum()})
res = pd.DataFrame(res).set_index(["version", "corte"])

def adopta(v):
    return (all(res.loc[(v, c), "gana_max"] >= 8 and res.loc[(v, c), "dif_nfijo"] > 0 for c in ["junio", "jun2m"])
            and res.loc[(v, "mayo"), "dif_nfijo"] >= -2)

candidatas = ["C_limpio", "BC_limpio"]
pasan = [v for v in candidatas if adopta(v)]
suma_nfijo = {v: res.loc[v, "dif_nfijo"].sum() for v in candidatas}
PARA_SUBMIT = pasan[0] if len(pasan) == 1 else max(suma_nfijo, key=suma_nfijo.get)
print("Pasan la regla:", pasan or "ninguna", "| suma dif N fijo:", {k: round(v, 1) for k, v in suma_nfijo.items()})
print("Versión para el submit 33:", PARA_SUBMIT)
log(f"pasan: {pasan} | submit 33: {PARA_SUBMIT}")
display(pd.DataFrame(curvas).pivot(index="version", columns="corte").round(1))
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
res.round(3)

Pasan la regla: ninguna | suma dif N fijo: {'C_limpio': np.float64(8.8), 'BC_limpio': np.float64(27.2)}
Versión para el submit 33: BC_limpio


max_curva_prom                n_max             
corte              jun2m  junio   mayo  jun2m  junio  mayo
version                                                   
BC_limpio          407.9  424.4  268.7  14874  14429  7264
C_limpio           401.6  414.4  269.2  15380  15186  6771
ref                401.0  415.9  271.3  14839  14989  6785

gan_max  gan_max_desv  gan_nfijo  gan_12000  mejor_n  dif_max  gana_max  p_max  dif_nfijo  gana_nfijo
version   corte                                                                                                       
ref       junio  418.828         3.443    387.502     387.75  14769.9    0.000       NaN    NaN       0.00         NaN
          mayo   274.466         1.980    250.580     250.14   8011.4    0.000       NaN    NaN       0.00         NaN
          jun2m  403.752         5.093    378.482     379.61  14758.1    0.000       NaN    NaN       0.00         NaN
C_limpio  junio  417.698         3.909    394.873     394.79  14995.0   -1.130       5.0  0.322       7.37         8.0
          mayo   272.308         1.451    248.380     247.94   8149.9   -2.159       2.0  0.027      -2.20         4.0
          jun2m  405.718         5.653    382.112     382.36  15022.6    1.966       6.0  0.492       3.63         5.0
BC_limpio junio  427.776         3.805    405.762     407.66  13892.5    8.948      10.0  0.002      18.26        10.0
          mayo   272.555         3.095    248.490     248.05   8184.9   -1.911       1.0  0.037      -2.09         4.0
          jun2m  411.455         5.709    389.482     389.84  15098.0    7.703       8.0  0.027      11.00         9.0

In [7]:
# --- 7. Anotar en resultados_experimentos.csv ---
ruta = "resultados_experimentos.csv"
hist = pd.read_csv(ruta, sep=";", decimal=",", encoding="utf-8-sig")
hist = hist[~hist["descripcion"].str.startswith("z027")]
nuevas_filas = []
for (version, corte), r in res.iterrows():
    meses_train, mes_valid = CORTES[corte]
    extra = "" if version == "ref" else (f"; dif máx {r['dif_max']:+.1f} M, gana {int(r['gana_max'])}/10, p={r['p_max']:.3f}; "
                                         f"dif N fijo {r['dif_nfijo']:+.1f} M, gana {int(r['gana_nfijo'])}/10; adopta: {'SÍ' if adopta(version) else 'no'}")
    nuevas_filas.append({
        "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
        "descripcion": f"z027 {version} ({corte}): trial 34, 10 semillas, {len(VERSIONES[version])} variables; N fijo {r['gan_nfijo']:.1f} M" + extra,
        "meses_train": str(meses_train), "mes_valid": mes_valid, "semilla": str(SEMILLAS), "params": str(params),
        "mejor_n": int(r["mejor_n"]), "ganancia_max_M": round(r["gan_max"], 1), "ganancia_top12000_M": round(r["gan_12000"], 1)})
hist = pd.concat([hist, pd.DataFrame(nuevas_filas)], ignore_index=True)
hist["n_enviados"] = hist["n_enviados"].astype("Int64")
hist.to_csv(ruta, sep=";", decimal=",", index=False, encoding="utf-8-sig")
log("todo listo")

## Conclusiones (11/10, 15:45) — ninguna pasa la regla; BC_limpio queda a 0,1 M de pasar

Diferencia contra la referencia, 10 semillas. Ganancia máx / N fijo (M), semillas ganadas (máx / N fijo):

| Versión | junio (04→06) | jun2m (03+04→06) | mayo (03→05) | Adopta |
|---|---|---|---|---|
| C_limpio | −1,1 / +7,4, 5/10 y 8/10 | +2,0 / +3,6, 6/10 y 5/10 | −2,2 / −2,2, 2/10 y 4/10 | **no** |
| BC_limpio | **+8,9 / +18,3, 10/10 y 10/10** (p=0,002) | **+7,7 / +11,0, 8/10 y 9/10** (p=0,03) | −1,9 / **−2,09**, 1/10 y 4/10 | **no** (mayo pasa el límite de −2 M por 0,09) |

Curva promedio (método del tutor), máx en M: junio ref 415,9 / BC 424,4 · jun2m ref 401,0 / BC 407,9 · mayo ref 271,3 / BC 268,7.

- **BC_limpio** gana fuerte en los dos cortes de junio, también con train de 2 meses (con historia). En mayo pierde poco (−2 M, ruido
  del orden de la desviación), pero de forma consistente (1/10 en máx). Sigue siendo la misma duda: mayo sin historia o efecto solo junio.
- **C_limpio** solo no alcanza: la mejora de z026 venía en buena parte de B.
- Según lo definido antes, la versión para el submit 33 es **BC_limpio** (mayor suma de diferencias en N fijo: +27,2 vs +8,8).
  Es una apuesta para ver en el bot, no algo validado: el ruido del público es ±12 M.
